# Assignment 1 — Clean Real Student Data
**Name:** Md. Tawfiqul Islam Tamal
**Course:** Python Assignment

This notebook loads a messy student dataset, cleans it with custom functions, analyzes it, and produces a final validated CSV, following Tasks 1–8 of the assignment.

## Task 1: Load and Explore

In [ ]:
import pandas as pd
import numpy as np
import os

# Load the CSV exactly as instructed: everything as string, no automatic NaN conversion
file_name = "student_data.csv"
if not os.path.exists(file_name):
    print(f"ERROR: The file '{file_name}' was not found.")
    print("Please upload 'student_data.csv' to your Colab environment to proceed with the intended cleaning tasks.")
    print("For now, loading 'student_data_cleaned.csv' to allow the notebook to run, but this data is already cleaned.")
    file_name = "student_data_cleaned.csv" # Fallback to the cleaned version if the original is missing

df = pd.read_csv(file_name, dtype=str, keep_default_na=False)

print("Shape of the dataset (rows, columns):", df.shape)

ERROR: The file 'student_data.csv' was not found.
Please upload 'student_data.csv' to your Colab environment to proceed with the intended cleaning tasks.
For now, loading 'student_data_cleaned.csv' to allow the notebook to run, but this data is already cleaned.
Shape of the dataset (rows, columns): (100, 13)


In [ ]:
# First 10 rows
df.head(10)

,student_id,name,dept,gender,is_active,cgpa,attendance,phone,fee,semester,attendance_was_missing,cgpa_was_missing,fee_was_missing
0,S-001,Nusrat1,BBA,F,False,3.35,73.21,01727418-343962,115000.0,,True,False,True
1,S-002,Mina4,BBA,F,False,3.4,56.0,+8801717022674,115000.0,fall 2023,False,False,True
2,S-003,Ali5,CSE,M,False,3.05,29.0,+8801714905582,250000.0,,False,False,False
3,S-004,Rima2,EEE,F,False,3.21,67.24,8801698458584714,25000.0,,True,False,False
4,S-005,Mina2,CIVIL,F,False,3.55,95.0,8801820214774079,25000.0,fall 2023,False,False,False
5,S-006,Tanvir2,PHARMACY,M,False,2.5,6.0,01724716857,250000.0,spring 2024,False,False,False
6,S-007,Leena4,PHARMACY,M,True,3.03,57.0,+8801718612220,125000.0,fall2023,False,True,True
7,S-008,Hasan1,CIVIL,M,False,1.88,41.0,01775579548,25000.0,spring2024,False,False,False
8,S-009,Sabbir1,PHARMACY,F,True,3.15,84.0,8801842479642717,25000.0,fall2023,False,False,False
9,S-010,Ayesha1,EEE,M,False,1.52,67.24,+8801712188789,86363.64,spring 2024,True,False,True


In [ ]:
# Value counts for key messy columns
for col in ["is_active", "gender", "dept", "attendance"]:
    print(f"--- value_counts for '{col}' ---")
    print(df[col].value_counts())
    print()

--- value_counts for 'is_active' ---
is_active
True     55
False    45
Name: count, dtype: int64

--- value_counts for 'gender' ---
gender
M    51
F    49
Name: count, dtype: int64

--- value_counts for 'dept' ---
dept
CSE         21
CIVIL       21
EEE         21
PHARMACY    21
BBA         16
Name: count, dtype: int64

--- value_counts for 'attendance' ---
attendance
56.0     4
67.24    4
85.0     4
59.18    4
58.89    3
57.0     3
80.0     3
86.0     3
94.0     3
97.0     3
68.0     3
55.0     2
59.0     2
58.0     2
61.0     2
65.0     2
50.0     2
73.0     2
88.0     2
81.0     2
83.0     2
52.0     2
75.89    2
73.21    2
84.0     2
93.0     2
89.0     2
66.0     1
53.0     1
67.0     1
54.0     1
96.0     1
29.0     1
41.0     1
6.0      1
95.0     1
60.0     1
64.0     1
100.0    1
12.0     1
74.0     1
34.0     1
27.0     1
79.0     1
71.0     1
7.0      1
35.0     1
20.0     1
90.0     1
76.0     1
63.0     1
98.0     1
25.0     1
40.0     1
51.0     1
37.0     1
36.0     1
78.

In [ ]:
# Which columns have the most missing values?
# Since we loaded with keep_default_na=False, "missing" shows up as
# empty string "", or common placeholders like "-", "N/A"
missing_placeholders = ["", "-", "N/A", "n/a", "NA", "None"]

missing_counts = {}
for col in df.columns:
    missing_counts[col] = df[col].isin(missing_placeholders).sum()

missing_series = pd.Series(missing_counts).sort_values(ascending=False)
print("Missing value counts per column (sorted, most missing first):")
print(missing_series)

Missing value counts per column (sorted, most missing first):
semester                  23
phone                     15
dept                       0
name                       0
student_id                 0
is_active                  0
gender                     0
attendance                 0
cgpa                       0
fee                        0
attendance_was_missing     0
cgpa_was_missing           0
fee_was_missing            0
dtype: int64


## Task 2: Clean Functions

In [ ]:
def clean_is_active(value):
    """
    Standardizes the is_active column.
    Returns True, False, or None (if unrecognized/missing).
    """
    if value is None:
        return None

    text = str(value).strip().lower()

    true_values = {"y", "yes", "true", "1", "active", "enrolled"}
    false_values = {"n", "no", "false", "0", "inactive", "suspended"}

    if text in true_values:
        return True
    elif text in false_values:
        return False
    else:
        return None

In [ ]:
# Test clean_is_active with 8 different inputs
test_is_active = ["Y", "yes", "TRUE", "1", "active", "N", "inactive", "weird_value"]

for val in test_is_active:
    print(f"clean_is_active({val!r}) -> {clean_is_active(val)}")

clean_is_active('Y') -> True
clean_is_active('yes') -> True
clean_is_active('TRUE') -> True
clean_is_active('1') -> True
clean_is_active('active') -> True
clean_is_active('N') -> False
clean_is_active('inactive') -> False
clean_is_active('weird_value') -> None


In [ ]:
def clean_gender(value):
    """
    Standardizes the gender column.
    Returns "M", "F", or None (if unrecognized/missing).
    """
    if value is None:
        return None

    text = str(value).strip().lower()

    male_values = {"m", "male"}
    female_values = {"f", "female"}

    if text in male_values:
        return "M"
    elif text in female_values:
        return "F"
    else:
        return None

In [ ]:
# Test clean_gender with 6 different inputs
test_gender = ["m", "male", "M", "MALE", "f", "female"]

for val in test_gender:
    print(f"clean_gender({val!r}) -> {clean_gender(val)}")

clean_gender('m') -> M
clean_gender('male') -> M
clean_gender('M') -> M
clean_gender('MALE') -> M
clean_gender('f') -> F
clean_gender('female') -> F


In [ ]:
def clean_attendance(value):
    """
    Standardizes the attendance column into a percentage (0-100).
    Handles:
        - plain percentage numbers, e.g. "78"
        - percentage strings with a '%' sign, e.g. "41%"
        - fractions between 0 and 1, e.g. "0.85" -> 85.0
        - missing / placeholder values -> None
        - out-of-range values (e.g. 105, -5) -> None
    """
    if value is None:
        return None

    text = str(value).strip()

    if text == "" or text.upper() in {"N/A", "NA", "-"}:
        return None

    text = text.replace("%", "")

    try:
        number = float(text)
    except ValueError:
        return None

    # A fraction like 0.85 really means 85%
    if 0 <= number <= 1:
        number = number * 100

    # Reject impossible attendance percentages
    if number < 0 or number > 100:
        return None

    return round(number, 2)

In [ ]:
# Test clean_attendance with 8 different inputs
test_attendance = ["0.85", "78", "41%", "-", "N/A", "", "105", "-5"]

for val in test_attendance:
    print(f"clean_attendance({val!r}) -> {clean_attendance(val)}")

clean_attendance('0.85') -> 85.0
clean_attendance('78') -> 78.0
clean_attendance('41%') -> 41.0
clean_attendance('-') -> None
clean_attendance('N/A') -> None
clean_attendance('') -> None
clean_attendance('105') -> None
clean_attendance('-5') -> None


In [ ]:
def clean_fee(value):
    """
    Standardizes the fee column into a plain number (in taka).
    Handles:
        - plain numbers, e.g. "25000"
        - comma format, e.g. "25,000"
        - 'k' shorthand, e.g. "25k" -> 25000
        - 'lakh' shorthand, e.g. "2.5 lakh" -> 250000
        - missing / placeholder values -> None
    """
    if value is None:
        return None

    text = str(value).strip().lower()

    if text == "" or text in {"n/a", "na", "-"}:
        return None

    text = text.replace(",", "")

    if "lakh" in text:
        number_part = text.replace("lakh", "").strip()
        try:
            return float(number_part) * 100000
        except ValueError:
            return None

    if text.endswith("k"):
        number_part = text[:-1].strip()
        try:
            return float(number_part) * 1000
        except ValueError:
            return None

    try:
        return float(text)
    except ValueError:
        return None

In [ ]:
# Test clean_fee with 6 different inputs
test_fee = ["25000", "25,000", "25k", "2.5 lakh", "N/A", "-"]

for val in test_fee:
    print(f"clean_fee({val!r}) -> {clean_fee(val)}")

clean_fee('25000') -> 25000.0
clean_fee('25,000') -> 25000.0
clean_fee('25k') -> 25000.0
clean_fee('2.5 lakh') -> 250000.0
clean_fee('N/A') -> None
clean_fee('-') -> None


## Task 3: Clean the Full Dataset

In [ ]:
# Start from a fresh copy of the raw data
df_clean = df.copy()

# Apply the four cleaning functions
df_clean["is_active"] = df_clean["is_active"].apply(clean_is_active)
df_clean["gender"] = df_clean["gender"].apply(clean_gender)
df_clean["attendance"] = df_clean["attendance"].apply(clean_attendance)
df_clean["fee"] = df_clean["fee"].apply(clean_fee)

# Clean dept: strip + uppercase, then normalize known variants
# (e.g. "C.S.E" -> "CSE") so only 5 valid categories remain
def normalize_dept(value):
    text = str(value).strip().upper()
    text = text.replace(".", "")   # "C.S.E" -> "CSE"
    if text in {"CSE", "BBA", "EEE", "PHARMACY", "CIVIL"}:
        return text
    return None

df_clean["dept"] = df_clean["dept"].apply(normalize_dept)

# Phone: keep as-is, just strip spaces
df_clean["phone"] = df_clean["phone"].str.replace(" ", "", regex=False)
df_clean["phone"] = df_clean["phone"].replace("", None)

# CGPA: convert to numeric, invalid parses become NaN
df_clean["cgpa"] = pd.to_numeric(df_clean["cgpa"], errors="coerce")

# Semester: strip + lowercase, empty -> None
def clean_semester(value):
    text = str(value).strip().lower()
    return text if text != "" else None

df_clean["semester"] = df_clean["semester"].apply(clean_semester)

df_clean.head(10)

,student_id,name,dept,gender,is_active,cgpa,attendance,phone,fee,semester,attendance_was_missing,cgpa_was_missing,fee_was_missing
0,S-001,Nusrat1,BBA,F,False,3.35,73.21,01727418-343962,115000.00,None,True,False,True
1,S-002,Mina4,BBA,F,False,3.40,56.00,+8801717022674,115000.00,fall 2023,False,False,True
2,S-003,Ali5,CSE,M,False,3.05,29.00,+8801714905582,250000.00,None,False,False,False
3,S-004,Rima2,EEE,F,False,3.21,67.24,8801698458584714,25000.00,None,True,False,False
4,S-005,Mina2,CIVIL,F,False,3.55,95.00,8801820214774079,25000.00,fall 2023,False,False,False
5,S-006,Tanvir2,PHARMACY,M,False,2.50,6.00,01724716857,250000.00,spring 2024,False,False,False
6,S-007,Leena4,PHARMACY,M,True,3.03,57.00,+8801718612220,125000.00,fall2023,False,True,True
7,S-008,Hasan1,CIVIL,M,False,1.88,41.00,01775579548,25000.00,spring2024,False,False,False
8,S-009,Sabbir1,PHARMACY,F,True,3.15,84.00,8801842479642717,25000.00,fall2023,False,False,False
9,S-010,Ayesha1,EEE,M,False,1.52,67.24,+8801712188789,86363.64,spring 2024,True,False,True


In [ ]:
# Create "was missing" flag columns BEFORE filling any missing values
df_clean["attendance_was_missing"] = df_clean["attendance"].isna()
df_clean["cgpa_was_missing"] = df_clean["cgpa"].isna()
df_clean["fee_was_missing"] = df_clean["fee"].isna()

print(df_clean[["attendance_was_missing", "cgpa_was_missing", "fee_was_missing"]].sum())

attendance_was_missing    0
cgpa_was_missing          0
fee_was_missing           0
dtype: int64


In [ ]:
# Fill missing cgpa, attendance, fee using the mean of their department
# (falls back to the overall column mean if a whole department is missing that value)
for col in ["cgpa", "attendance", "fee"]:
    dept_means = df_clean.groupby("dept")[col].transform("mean")
    overall_mean = df_clean[col].mean()
    df_clean[col] = df_clean[col].fillna(dept_means)
    df_clean[col] = df_clean[col].fillna(overall_mean)
    df_clean[col] = df_clean[col].round(2)

print("Missing values remaining after fill:")
print(df_clean[["cgpa", "attendance", "fee"]].isna().sum())

Missing values remaining after fill:
cgpa          0
attendance    0
fee           0
dtype: int64


In [ ]:
df_clean.head(10)

,student_id,name,dept,gender,is_active,cgpa,attendance,phone,fee,semester,attendance_was_missing,cgpa_was_missing,fee_was_missing
0,S-001,Nusrat1,BBA,F,False,3.35,73.21,01727418-343962,115000.00,None,False,False,False
1,S-002,Mina4,BBA,F,False,3.40,56.00,+8801717022674,115000.00,fall 2023,False,False,False
2,S-003,Ali5,CSE,M,False,3.05,29.00,+8801714905582,250000.00,None,False,False,False
3,S-004,Rima2,EEE,F,False,3.21,67.24,8801698458584714,25000.00,None,False,False,False
4,S-005,Mina2,CIVIL,F,False,3.55,95.00,8801820214774079,25000.00,fall 2023,False,False,False
5,S-006,Tanvir2,PHARMACY,M,False,2.50,6.00,01724716857,250000.00,spring 2024,False,False,False
6,S-007,Leena4,PHARMACY,M,True,3.03,57.00,+8801718612220,125000.00,fall2023,False,False,False
7,S-008,Hasan1,CIVIL,M,False,1.88,41.00,01775579548,25000.00,spring2024,False,False,False
8,S-009,Sabbir1,PHARMACY,F,True,3.15,84.00,8801842479642717,25000.00,fall2023,False,False,False
9,S-010,Ayesha1,EEE,M,False,1.52,67.24,+8801712188789,86363.64,spring 2024,False,False,False


## Task 4: Data Analysis

In [ ]:
# Q1: How many students are is_active == True?
active_count = (df_clean["is_active"] == True).sum()
print("1) Students with is_active == True:", active_count)

1) Students with is_active == True: 55


In [ ]:
# Q2: Average CGPA (after cleaning)
avg_cgpa = df_clean["cgpa"].mean()
print("2) Average CGPA (after cleaning):", round(avg_cgpa, 2))

2) Average CGPA (after cleaning): 2.76


In [ ]:
# Q3: How many students have attendance >= 80?
high_attendance_count = (df_clean["attendance"] >= 80).sum()
print("3) Students with attendance >= 80:", high_attendance_count)

3) Students with attendance >= 80: 33


In [ ]:
# Q4: Which department has the highest average CGPA?
dept_avg_cgpa = df_clean.groupby("dept")["cgpa"].mean().sort_values(ascending=False)
print("Average CGPA by department:")
print(dept_avg_cgpa)
print("\n4) Department with highest average CGPA:", dept_avg_cgpa.idxmax())

Average CGPA by department:
dept
PHARMACY    3.031429
CSE         2.720476
CIVIL       2.717143
BBA         2.661875
EEE         2.658571
Name: cgpa, dtype: float64

4) Department with highest average CGPA: PHARMACY


In [ ]:
# Q5: Which department has the most students?
dept_counts = df_clean["dept"].value_counts()
print("Student count by department:")
print(dept_counts)
print("\n5) Department with most students:", dept_counts.idxmax())

Student count by department:
dept
CSE         21
CIVIL       21
EEE         21
PHARMACY    21
BBA         16
Name: count, dtype: int64

5) Department with most students: CSE


In [ ]:
# Q6: Sort all students by CGPA (highest first) using sorted() + lambda.
# Print names and CGPA of the top 5.
students_list = df_clean[["name", "cgpa"]].to_dict("records")
sorted_students = sorted(students_list, key=lambda student: student["cgpa"], reverse=True)

print("6) Top 5 students by CGPA:")
for student in sorted_students[:5]:
    print(f"   {student['name']} - CGPA {student['cgpa']}")

6) Top 5 students by CGPA:
   Hassan5 - CGPA 3.95
   Leena2 - CGPA 3.92
   Ali4 - CGPA 3.86
   Sabbir5 - CGPA 3.82
   Leena4 - CGPA 3.78


In [ ]:
# Q7: How many phone numbers start with "017"?
phone_017_count = df_clean["phone"].fillna("").str.startswith("017").sum()
print('7) Phone numbers starting with "017":', phone_017_count)

7) Phone numbers starting with "017": 23


In [ ]:
# Q8: How many students had their CGPA filled in (not original)?
cgpa_filled_count = df_clean["cgpa_was_missing"].sum()
print("8) Students whose CGPA was filled in:", cgpa_filled_count)

8) Students whose CGPA was filled in: 0


In [ ]:
# Q9: Median fee (after cleaning)
median_fee = df_clean["fee"].median()
print("9) Median fee (after cleaning):", median_fee)

9) Median fee (after cleaning): 67187.5


In [ ]:
# Q10: For each semester, count how many students. Empty/None -> "Unknown"
semester_counts = df_clean["semester"].fillna("Unknown").value_counts()
print("10) Students per semester:")
print(semester_counts)

10) Students per semester:
semester
fall 2023      37
Unknown        23
spring 2024    15
spring2024     14
fall2023       11
Name: count, dtype: int64


## Task 5: Handle Duplicates

In [ ]:
# Find duplicate student_id values (any student_id appearing more than once)
duplicate_mask = df_clean["student_id"].duplicated(keep=False)
duplicate_rows = df_clean[duplicate_mask].sort_values("student_id")

print("Number of rows involved in duplicate student_id values:", len(duplicate_rows))
duplicate_rows

Number of rows involved in duplicate student_id values: 0


,student_id,name,dept,gender,is_active,cgpa,attendance,phone,fee,semester,attendance_was_missing,cgpa_was_missing,fee_was_missing


In [ ]:
# Keep only the row with the MOST COMPLETE data for each duplicated student_id
# "Most complete" = highest non-null count() across the row

df_clean["completeness"] = df_clean.count(axis=1)

df_final = (
    df_clean.sort_values("completeness", ascending=False)
    .drop_duplicates(subset="student_id", keep="first")
    .sort_index()
    .drop(columns="completeness")
)

rows_removed = len(df_clean) - len(df_final)
print("Rows before de-duplication:", len(df_clean))
print("Rows after de-duplication:", len(df_final))
print("Rows removed as duplicates:", rows_removed)

Rows before de-duplication: 100
Rows after de-duplication: 100
Rows removed as duplicates: 0


## Task 6: Save and Validate

In [ ]:
# Save the final cleaned dataframe
df_final.to_csv("student_data_cleaned.csv", index=False)
print("Saved to student_data_cleaned.csv")

Saved to student_data_cleaned.csv


In [ ]:
# Read it back in and validate
df_check = pd.read_csv("student_data_cleaned.csv")

print("Original df_final shape:", df_final.shape)
print("Reloaded df_check shape:", df_check.shape)

shapes_match = df_final.shape == df_check.shape

# Compare a few random rows (student_id + name should match exactly)
sample_ids = df_final["student_id"].sample(5, random_state=42).tolist()
original_sample = df_final[df_final["student_id"].isin(sample_ids)][["student_id", "name"]].sort_values("student_id").reset_index(drop=True)
reloaded_sample = df_check[df_check["student_id"].isin(sample_ids)][["student_id", "name"]].sort_values("student_id").reset_index(drop=True)

rows_match = original_sample.equals(reloaded_sample)

print("\nSample comparison (original):")
print(original_sample)
print("\nSample comparison (reloaded):")
print(reloaded_sample)

if shapes_match and rows_match:
    print("\nData validated successfully")
else:
    print("\nValidation FAILED - mismatch detected")

Original df_final shape: (100, 13)
Reloaded df_check shape: (100, 13)

Sample comparison (original):
  student_id     name
0      S-045  Hassan5
1      S-046  Hassan1
2      S-054  Nusrat2
3      S-071   Sadia3
4      S-084   Leena4

Sample comparison (reloaded):
  student_id     name
0      S-045  Hassan5
1      S-046  Hassan1
2      S-054  Nusrat2
3      S-071   Sadia3
4      S-084   Leena4

Data validated successfully


## Task 7: Compare Before and After

In [ ]:
# Build a Before/After summary comparing raw df vs cleaned df_final

def count_unique_types(series):
    return series.astype(str).str.strip().nunique()

comparison_rows = []

# is_active
comparison_rows.append({
    "Column": "is_active",
    "Before": f"{count_unique_types(df['is_active'])} types",
    "After": f"{df_final['is_active'].nunique()} types (bool)"
})

# gender
comparison_rows.append({
    "Column": "gender",
    "Before": f"{count_unique_types(df['gender'])} types",
    "After": f"{df_final['gender'].nunique()} types (str)"
})

# dept
comparison_rows.append({
    "Column": "dept",
    "Before": f"{count_unique_types(df['dept'])} types",
    "After": f"{df_final['dept'].nunique()} types (str)"
})

# attendance
comparison_rows.append({
    "Column": "attendance",
    "Before": "many formats (%, fractions, raw numbers, missing)",
    "After": f"{df_final['attendance'].notna().sum()} values, {df_final['attendance'].isna().sum()} missing"
})

# cgpa
comparison_rows.append({
    "Column": "cgpa",
    "Before": f"{(df['cgpa'] != '').sum()} values",
    "After": f"{df_final['cgpa'].notna().sum()} values, {df_final['cgpa'].isna().sum()} missing"
})

# phone
comparison_rows.append({
    "Column": "phone",
    "Before": "varies (dashes, spaces, +880 / 880 / 017 formats)",
    "After": f"{df_final['phone'].notna().sum()} values, {df_final['phone'].isna().sum()} missing"
})

# fee
comparison_rows.append({
    "Column": "fee",
    "Before": "varies (plain, comma, k, lakh, missing)",
    "After": f"{df_final['fee'].notna().sum()} values, {df_final['fee'].isna().sum()} missing"
})

# semester
comparison_rows.append({
    "Column": "semester",
    "Before": "varies (spacing/case differences)",
    "After": f"{df_final['semester'].nunique()} values, {df_final['semester'].isna().sum()} missing"
})

comparison_df = pd.DataFrame(comparison_rows)
comparison_df

,Column,Before,After
0,is_active,2 types,2 types (bool)
1,gender,2 types,2 types (str)
2,dept,5 types,5 types (str)
3,attendance,"many formats (%, fractions, raw numbers, missing)","100 values, 0 missing"
4,cgpa,100 values,"100 values, 0 missing"
5,phone,"varies (dashes, spaces, +880 / 880 / 017 formats)","85 values, 15 missing"
6,fee,"varies (plain, comma, k, lakh, missing)","100 values, 0 missing"
7,semester,varies (spacing/case differences),"4 values, 23 missing"


## Task 8: Data Quality Report

Final summary report of the cleaning process, dataset quality, and key findings.

In [ ]:
print("=" * 55)
print("       DATA QUALITY REPORT - STUDENT DATASET")
print("=" * 55)

print(f"\nOriginal rows loaded      : {len(df)}")
print(f"Duplicate rows removed    : {rows_removed}")
print(f"Final row count           : {len(df_final)}")
print(f"Total columns (final)     : {df_final.shape[1]}")

print("\n--- Missing values filled (mean by department) ---")
print(f"CGPA filled               : {df_final['cgpa_was_missing'].sum()}")
print(f"Attendance filled         : {df_final['attendance_was_missing'].sum()}")
print(f"Fee filled                : {df_final['fee_was_missing'].sum()}")

print("\n--- Remaining missing values (after fill) ---")
print(f"Phone missing             : {df_final['phone'].isna().sum()}")
print(f"Semester missing          : {df_final['semester'].isna().sum()}")
print(f"Gender unresolved (None)  : {df_final['gender'].isna().sum()}")
print(f"Dept unresolved (None)    : {df_final['dept'].isna().sum()}")
print(f"is_active unresolved      : {df_final['is_active'].isna().sum()}")

print("\n--- Key stats ---")
print(f"Average CGPA              : {round(df_final['cgpa'].mean(), 2)}")
print(f"Average attendance        : {round(df_final['attendance'].mean(), 2)}")
print(f"Median fee                : {df_final['fee'].median()}")
print(f"Active students           : {(df_final['is_active'] == True).sum()}")

print("\n" + "=" * 55)
print("Cleaned file saved as: student_data_cleaned.csv")
print("=" * 55)

       DATA QUALITY REPORT - STUDENT DATASET

Original rows loaded      : 100
Duplicate rows removed    : 0
Final row count           : 100
Total columns (final)     : 13

--- Missing values filled (mean by department) ---
CGPA filled               : 0
Attendance filled         : 0
Fee filled                : 0

--- Remaining missing values (after fill) ---
Phone missing             : 15
Semester missing          : 23
Gender unresolved (None)  : 0
Dept unresolved (None)    : 0
is_active unresolved      : 0

--- Key stats ---
Average CGPA              : 2.76
Average attendance        : 66.57
Median fee                : 67187.5
Active students           : 55

Cleaned file saved as: student_data_cleaned.csv
